# Figure 3 recovery: reuse F1/C1, compute C2 only (restart-safe)

**Important:** First save F1/C1 from the *old live Colab kernel* using the rescue cell provided in the chat. This notebook **will not re-run F1/C1** and **will never retrain any model**. It requires these existing Drive files:

- `F1_freshly_measured_curve.csv` and `C1_freshly_measured_curve.csv`, containing the 61 measurements actually evaluated earlier
- All six `F1/C1/C2_SB/LB_best.pt` files, trained and saved by the original notebook

Switch Colab to **T4 GPU** before running this notebook. The cells below define the original exact model, preprocessing, and evaluation routines; evaluate only C2 directly from the saved C2 endpoints; save each C2 α measurement to a Drive CSV immediately; then plot all three measured curves. After a disconnect, **Run all** again: existing, valid C2 points are skipped. No measured values are fabricated, digitized, or hardcoded. Keep CONFIG and its checkpoint-relevant options unchanged.

The C2 `.csv` is a durable *cache of measurements computed from checkpoints*; F1/C1 `.csv`s are the same kind of actual-run results from the earlier live session. This is a recovery workflow: plotting uses the saved measured values rather than reevaluating F1/C1.


In [ ]:
# Cell 1 — Configuration: all experimentally chosen numbers live here, not in the plotting code.
from pathlib import Path

CONFIG = {
    "seed": 1729,
    "networks": ("F1", "C1", "C2"),
    "use_google_drive": True,       # For persistent checkpoints across Colab sessions.
    "drive_dir": "/content/drive/MyDrive/keskar_fig3_f1_c1_c2",
    "local_dir": "/content/keskar_fig3_f1_c1_c2",
    "data_dir": "/content/keskar_fig3_datasets",
    "num_workers": 1,             # Keeps Colab shared-memory use low for 5000-image batches.
    "pin_memory": True,
    "lr": 1e-3,
    "adam_betas": (0.9, 0.999),
    "adam_eps": 1e-8,
    "weight_decay": 0.0,
    "max_epochs": 100,              # Computational safety cap; NOT specified in paper.
    "patience": 15,                # Consecutive epochs without a significant CE drop.
    "min_delta": 1e-4,             # Plateau threshold in CE units.
    "save_every_epochs": 1,         # Resume progress in completed epochs.
    "bn_calibration_examples": 2048,# Fixed training subset; NOT specified in paper.
    "alpha_min": -1.0,
    "alpha_sb": 0.0,
    "alpha_lb": 1.0,
    "alpha_max": 2.0,
    "alpha_grid_points": 61,        # includes SB/LB endpoints with these bounds.
    "input_divisor": 255.0,        # uint8 pixels -> float32 in [0,1].
    "test_run_synthetic_checks": True,
    "models": {
        "F1": {"dataset": "MNIST",    "small_batch": 256, "large_batch": 6000,
               "physical_batch": 256, "eval_batch": 1024, "calib_batch": 256},
        "C1": {"dataset": "CIFAR10", "small_batch": 256, "large_batch": 5000,
               "physical_batch": 128, "eval_batch": 256,  "calib_batch": 128},
        "C2": {"dataset": "CIFAR10", "small_batch": 256, "large_batch": 5000,
               "physical_batch": 32,  "eval_batch": 64,   "calib_batch": 32},
    },
}

PLOT_STYLE = {             # Plot layout/format configuration; NO fixed metric data.
    "rows": 1,
    "figsize": (18, 5),
    "dpi": 180,
    "line_width": 1.7,
    "train_ls": "-",
    "test_ls": "--",
    "loss_color": "#225bb3",
    "accuracy_color": "#ca3030",
    "accuracy_ylim": (0, 100),
    "grid_alpha": 0.35,
    "file_name": "figure3_F1_C1_C2_computed.png",
}

In [ ]:
# Cell 2 — Imports, GPU environment, persistent output directory, seeds.
import math, json, hashlib, random, time, copy, gc, csv
from collections import OrderedDict

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision.datasets import MNIST, CIFAR10
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

if CONFIG["use_google_drive"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = Path(CONFIG["drive_dir"])
    except ImportError:
        print("Outside Colab; using local directory instead of Google Drive.")
        OUT_DIR = Path(CONFIG["local_dir"])
else:
    OUT_DIR = Path(CONFIG["local_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path(CONFIG["data_dir"])
DATA_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(DEVICE))
else:
    print("WARNING: no CUDA GPU detected. Full training is impractical on CPU.")

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])
# FP32 throughout: no automatic mixed precision and no TF32 approximation.
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False

print("PyTorch:", torch.__version__, " | Artifacts:", OUT_DIR)

In [ ]:
# Cell 3 — Define F1/C1/C2 directly from the PDF's layer descriptions.
class F1Network(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        in_features = 784
        for _ in range(5):
            layers += [nn.Linear(in_features, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True)]
            in_features = 512
        self.hidden = nn.Sequential(*layers)
        self.classifier = nn.Linear(512, 10)

    def forward(self, x):
        x = x.flatten(start_dim=1)
        return self.classifier(self.hidden(x))


class C1Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        # For 32x32 input with the *specified/assumed* strides: [64,1,1].
        self.hidden = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 384), nn.BatchNorm1d(384), nn.ReLU(inplace=True), nn.Dropout(p=0.5),
            nn.Linear(384, 192), nn.BatchNorm1d(192), nn.ReLU(inplace=True), nn.Dropout(p=0.5),
        )
        self.classifier = nn.Linear(192, 10)

    def forward(self, x):
        return self.classifier(self.hidden(self.features(x)))


class C2Network(nn.Module):
    def __init__(self):
        super().__init__()
        # The PDF calls for 2,2,3,3,3 stride-1 3x3 convs, 2x2 pool after each block.
        blocks = []
        in_channels = 3
        for out_channels, depth in [(64, 2), (128, 2), (256, 3), (512, 3), (512, 3)]:
            for _ in range(depth):
                blocks += [
                    nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
                    nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
                ]
                in_channels = out_channels
            blocks.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.features = nn.Sequential(*blocks)
        # 32x32 -> 1x1 after five max pools; 512-channel flat feature.
        self.hidden = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True), nn.Dropout(p=0.5),
        )
        self.classifier = nn.Linear(512, 10)

    def forward(self, x):
        return self.classifier(self.hidden(self.features(x)))


def new_model(name):
    return {"F1": F1Network, "C1": C1Network, "C2": C2Network}[name]()


def get_dataset(dataset_name):
    '''Return (train, test) as TensorDatasets, retaining original uint8 bytes.'''
    if dataset_name == "MNIST":
        train_raw = MNIST(root=DATA_DIR, train=True, download=True)
        test_raw = MNIST(root=DATA_DIR, train=False, download=True)
        def make(raw):
            return TensorDataset(raw.data.unsqueeze(1).contiguous(),
                                 torch.as_tensor(raw.targets, dtype=torch.long))
    elif dataset_name == "CIFAR10":
        train_raw = CIFAR10(root=DATA_DIR, train=True, download=True)
        test_raw = CIFAR10(root=DATA_DIR, train=False, download=True)
        def make(raw):
            # CIFAR-10 data is NHWC uint8; convert to NCHW without normalization.
            x = torch.from_numpy(raw.data).permute(0, 3, 1, 2).contiguous()
            return TensorDataset(x, torch.as_tensor(raw.targets, dtype=torch.long))
    else:
        raise ValueError(dataset_name)
    return make(train_raw), make(test_raw)

DATASETS = {}  # Lazy cache; CIFAR reused by C1 and C2.

def datasets_for(name):
    key = CONFIG["models"][name]["dataset"]
    if key not in DATASETS:
        DATASETS[key] = get_dataset(key)
    return DATASETS[key]


def loader_for(dataset, batch_size, *, shuffle=False, generator=None):
    num_workers = CONFIG["num_workers"]
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, generator=generator,
        num_workers=num_workers,
        pin_memory=(CONFIG["pin_memory"] and DEVICE.type == "cuda"),
        persistent_workers=(num_workers > 0),
        **({"prefetch_factor": 1} if num_workers > 0 else {}),
    )


def to_device_pixels(x_uint8, targets):
    x = x_uint8.to(DEVICE, non_blocking=True).to(torch.float32)
    x.div_(CONFIG["input_divisor"])
    y = targets.to(DEVICE, non_blocking=True)
    return x, y

In [ ]:
# Cell 4 — Actual full-dataset measurements, training, and resumable checkpoint manager.
def model_initial_seed(name):
    # Independent starts BETWEEN model families; IDENTICAL start WITHIN SB/LB pair.
    return CONFIG["seed"] + {"F1": 0, "C1": 10000, "C2": 20000}[name]


def initialize_identically(name):
    seed = model_initial_seed(name)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return new_model(name).to(DEVICE)


def initial_digest(model):
    sha = hashlib.sha256()
    for key, param in model.named_parameters():
        sha.update(key.encode("utf-8"))
        sha.update(param.detach().cpu().contiguous().numpy().tobytes())
    return sha.hexdigest()


def training_signature(name, regime):
    # Excludes max_epochs so a capped run can be extended across Colab sessions.
    c = CONFIG["models"][name]
    b = c["small_batch"] if regime == "SB" else c["large_batch"]
    setting = {"name": name, "regime": regime, "effective_batch": b,
               "physical_batch": c["physical_batch"],
               "model_initial_seed": model_initial_seed(name),
               "lr": CONFIG["lr"], "betas": CONFIG["adam_betas"],
               "eps": CONFIG["adam_eps"], "weight_decay": CONFIG["weight_decay"],
               "patience": CONFIG["patience"], "min_delta": CONFIG["min_delta"],
               "input_divisor": CONFIG["input_divisor"]}
    return hashlib.sha256(json.dumps(setting, sort_keys=True).encode()).hexdigest()


def ckpt_path(name, regime, purpose):
    return OUT_DIR / f"{name}_{regime}_{purpose}.pt"


def save_atomic(payload, path):
    temp_path = path.with_suffix(path.suffix + ".partial")
    torch.save(payload, temp_path)
    temp_path.replace(path)


def torch_load_own(path):
    # These are locally generated checkpoints containing tensors and primitive metadata.
    return torch.load(path, map_location="cpu", weights_only=True)


@torch.inference_mode()
def measure_dataset(model, data_loader):
    '''Exact sample-weighted mean CE and top-1 accuracy on the entire supplied loader.'''
    model.eval()
    summed_loss, summed_correct, count = 0.0, 0, 0
    for raw, y_cpu in data_loader:
        x, y = to_device_pixels(raw, y_cpu)
        logits = model(x)
        summed_loss += F.cross_entropy(logits, y, reduction="sum").item()
        summed_correct += int((logits.argmax(dim=1) == y).sum().item())
        count += y.numel()
    assert count == len(data_loader.dataset)
    return {"ce": summed_loss / count, "accuracy": 100.0 * summed_correct / count,
            "examples": count}


def train_one(name, regime):
    config_model = CONFIG["models"][name]
    train_data, _ = datasets_for(name)
    effective_batch = (config_model["small_batch"] if regime == "SB"
                       else config_model["large_batch"])
    physical_batch = config_model["physical_batch"]
    signature = training_signature(name, regime)
    best_file = ckpt_path(name, regime, "best")
    progress_file = ckpt_path(name, regime, "progress")

    model = initialize_identically(name)
    starting_digest = initial_digest(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"],
                                 betas=CONFIG["adam_betas"], eps=CONFIG["adam_eps"],
                                 weight_decay=CONFIG["weight_decay"])
    first_epoch, best_loss, stale = 1, float("inf"), 0
    if progress_file.exists():
        progress = torch_load_own(progress_file)
        if progress["signature"] != signature or progress["initial_digest"] != starting_digest:
            raise RuntimeError(f"Stored progress settings/initialization mismatch: {progress_file}")
        model.load_state_dict(progress["model_state"])
        optimizer.load_state_dict(progress["optimizer_state"])
        # Loading to GPU model sometimes leaves optimizer buffers on CPU; move them explicitly.
        for state in optimizer.state.values():
            for key, value in state.items():
                if isinstance(value, torch.Tensor):
                    state[key] = value.to(DEVICE)
        first_epoch = progress["epoch"] + 1
        best_loss, stale = progress["best_loss"], progress["stale"]
        torch.set_rng_state(progress["cpu_rng"])
        if DEVICE.type == "cuda" and progress.get("cuda_rng") is not None:
            torch.cuda.set_rng_state(progress["cuda_rng"], device=DEVICE)
        if progress.get("stop_reason") == "plateau":
            if not best_file.exists():
                raise FileNotFoundError("Progress says complete but best weights are missing.")
            print(f"{name} {regime}: plateau reached at epoch {progress['epoch']}; using {best_file.name}")
            del model, optimizer
            return best_file
        print(f"{name} {regime}: resuming at epoch {first_epoch}")
    if first_epoch > CONFIG["max_epochs"]:
        if not best_file.exists():
            raise FileNotFoundError(f"Missing best checkpoint: {best_file}")
        print(f"{name} {regime}: reached epoch cap; using {best_file.name}")
        del model, optimizer
        return best_file

    eval_train_loader = loader_for(train_data, config_model["eval_batch"])
    for epoch in range(first_epoch, CONFIG["max_epochs"] + 1):
        tic = time.perf_counter()
        # Epoch-dependent deterministic shuffle, independent of DataLoader worker scheduling.
        shuffle_seed = model_initial_seed(name) + (0 if regime == "SB" else 1000000) + epoch
        epoch_gen = torch.Generator().manual_seed(shuffle_seed)
        train_loader = loader_for(train_data, effective_batch, shuffle=True, generator=epoch_gen)
        model.train()
        epoch_updates = 0
        seen = 0
        for raw_batch, y_batch in train_loader:
            batch_count = len(y_batch)
            # tensor_split makes nearly equal chunks: avoids a BN chunk of size ~1.
            n_chunks = math.ceil(batch_count / physical_batch)
            raw_parts = torch.tensor_split(raw_batch, n_chunks)
            y_parts = torch.tensor_split(y_batch, n_chunks)
            optimizer.zero_grad(set_to_none=True)
            for x_raw, y_raw in zip(raw_parts, y_parts):
                x, y = to_device_pixels(x_raw, y_raw)
                logits = model(x)
                (F.cross_entropy(logits, y, reduction="sum") / batch_count).backward()
            optimizer.step()
            epoch_updates += 1
            seen += batch_count
        assert seen == len(train_data)
        del train_loader
        exact_train = measure_dataset(model, eval_train_loader)
        epoch_loss = exact_train["ce"]
        if not math.isfinite(epoch_loss):
            raise FloatingPointError(f"Nonfinite training CE at {name} {regime}, epoch {epoch}.")
        improvement = (best_loss - epoch_loss) > CONFIG["min_delta"]
        if improvement:
            best_loss, stale = epoch_loss, 0
            save_atomic({"model_state": {k: v.detach().cpu().clone()
                                          for k, v in model.state_dict().items()},
                         "network": name, "regime": regime, "epoch": epoch,
                         "training_ce": epoch_loss, "training_accuracy": exact_train["accuracy"],
                         "signature": signature, "initial_digest": starting_digest}, best_file)
        else:
            stale += 1
        stop_reason = ("plateau" if stale >= CONFIG["patience"]
                       else "epoch_cap" if epoch >= CONFIG["max_epochs"] else None)
        if epoch % CONFIG["save_every_epochs"] == 0 or stop_reason is not None:
            save_atomic({"model_state": {k: v.detach().cpu().clone()
                                          for k, v in model.state_dict().items()},
                         "optimizer_state": optimizer.state_dict(),
                         "signature": signature, "initial_digest": starting_digest,
                         "epoch": epoch, "best_loss": best_loss, "stale": stale,
                         "stop_reason": stop_reason,
                         "cpu_rng": torch.get_rng_state(),
                         "cuda_rng": torch.cuda.get_rng_state(DEVICE)
                                     if DEVICE.type == "cuda" else None}, progress_file)
        print(f"{name:2s} {regime} | epoch {epoch:3d}/{CONFIG['max_epochs']} "
              f"| {epoch_updates:3d} optimizer steps | train CE {epoch_loss:.5f} "
              f"| train acc {exact_train['accuracy']:.2f}% "
              f"| best CE {best_loss:.5f} | plateau {stale}/{CONFIG['patience']} "
              f"| {time.perf_counter()-tic:.1f}s", flush=True)
        if stop_reason is not None:
            print(f"{name} {regime}: stopped due to {stop_reason}")
            break
    if not best_file.exists():
        raise FileNotFoundError("Training ended before any best model was saved.")
    del model, optimizer, eval_train_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return best_file

In [ ]:
# Cell 7 — Load saved endpoints, interpolate trained weights, recalibrate BN, measure ALL samples.
def fixed_calibration_subset(train_dataset, network_name):
    n = min(CONFIG["bn_calibration_examples"], len(train_dataset))
    gen = torch.Generator().manual_seed(CONFIG["seed"] + 333333)
    indices = torch.randperm(len(train_dataset), generator=gen)[:n].tolist()
    return Subset(train_dataset, indices)


@torch.no_grad()
def interpolate_parameters_(model, small_params, large_params, alpha):
    '''Apply theta(alpha) = (1-alpha)*small + alpha*large to TRAINABLE parameters.

    States are on the current device; BN running buffers are purposely excluded.
    torch.lerp also supports extrapolation (alpha outside [0,1]).
    '''
    for key, param in model.named_parameters():
        param.copy_(torch.lerp(small_params[key], large_params[key], float(alpha)))


@torch.no_grad()
def recalibrate_batchnorm(model, calibration_loader):
    # IMPORTANT: model.eval() disables dropout; only the BN modules are reenabled.
    model.eval()
    bn_modules = [module for module in model.modules()
                  if isinstance(module, nn.modules.batchnorm._BatchNorm)]
    original_momenta = [module.momentum for module in bn_modules]
    for module in bn_modules:
        assert module.track_running_stats, "This protocol requires BN running statistics."
        module.reset_running_stats()
        module.momentum = None  # Cumulative (equal-batch-weighted) running statistics.
        module.train()          # Activates updates on calibration images, but NO dropout.
    for images_raw, target in calibration_loader:
        x, _ = to_device_pixels(images_raw, target)
        model(x)                # No gradient; do not use labels for calibration.
    model.eval()
    for module, momentum in zip(bn_modules, original_momenta):
        module.momentum = momentum


# All buffer keys are derived from module structure, not a hardcoded key-pattern guess.
def get_bn_buffer_keys(model):
    buffer_keys = set()
    for module_name, module in model.named_modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            for key, _value in module.named_buffers(recurse=False):
                buffer_keys.add(f"{module_name}.{key}" if module_name else key)
    return buffer_keys

def read_and_validate_endpoints(name, model):
    small_file = ckpt_path(name, "SB", "best")
    large_file = ckpt_path(name, "LB", "best")
    if not small_file.exists() or not large_file.exists():
        raise FileNotFoundError(f"Train BOTH {name} endpoints first: {small_file}, {large_file}")
    s, l = torch_load_own(small_file), torch_load_own(large_file)
    assert s["network"] == l["network"] == name
    assert s["regime"] == "SB" and l["regime"] == "LB"
    assert s["signature"] == training_signature(name, "SB")
    assert l["signature"] == training_signature(name, "LB")
    assert s["initial_digest"] == l["initial_digest"], (
        "SB and LB must begin from identical initialization!")
    keys = set(dict(model.named_parameters()))
    assert keys == set(s["model_state"]) - get_bn_buffer_keys(model)
    assert keys == set(l["model_state"]) - get_bn_buffer_keys(model)
    return s, l


def compute_curve_directly_from_checkpoints(name):
    '''Called by the plotting cell: every point is evaluated now, not read from CSV.'''
    model = new_model(name).to(DEVICE)
    sb, lb = read_and_validate_endpoints(name, model)
    train_data, test_data = datasets_for(name)
    cfg = CONFIG["models"][name]
    train_loader = loader_for(train_data, cfg["eval_batch"])
    test_loader = loader_for(test_data, cfg["eval_batch"])
    cal_subset = fixed_calibration_subset(train_data, name)
    cal_loader = loader_for(cal_subset, cfg["calib_batch"])

    param_keys = set(dict(model.named_parameters()))
    small_params = {k: sb["model_state"][k].to(DEVICE) for k in param_keys}
    large_params = {k: lb["model_state"][k].to(DEVICE) for k in param_keys}

    # Checkpoint metric diagnostics with ORIGINAL endpoint BN buffers; not used for plotting.
    original_endpoint_metrics = {}
    for regime, checkpoint in (("SB", sb), ("LB", lb)):
        model.load_state_dict(checkpoint["model_state"])
        original_endpoint_metrics[regime] = {
            "train": measure_dataset(model, train_loader),
            "test": measure_dataset(model, test_loader),
        }
    # The grid automatically includes 0 and 1 even if an alternative density is chosen.
    alphas = np.unique(np.concatenate([
        np.linspace(CONFIG["alpha_min"], CONFIG["alpha_max"], CONFIG["alpha_grid_points"]),
        np.asarray([CONFIG["alpha_sb"], CONFIG["alpha_lb"]], dtype=np.float64),
    ]))
    result = []
    print(f"{name}: recalibrating BN and measuring complete datasets at {len(alphas)} alpha values.")
    begin = time.perf_counter()
    for k, alpha in enumerate(alphas):
        with torch.no_grad():
            interpolate_parameters_(model, small_params, large_params, alpha)
        recalibrate_batchnorm(model, cal_loader)
        train_metrics = measure_dataset(model, train_loader)
        test_metrics = measure_dataset(model, test_loader)
        record = {"network": name, "alpha": float(alpha),
                  "train_ce": train_metrics["ce"], "test_ce": test_metrics["ce"],
                  "train_acc": train_metrics["accuracy"], "test_acc": test_metrics["accuracy"]}
        assert all(math.isfinite(record[key])
                   for key in ("train_ce", "test_ce", "train_acc", "test_acc"))
        result.append(record)
        print(f"{name} {k+1:3d}/{len(alphas)} alpha={alpha:+.3f} "
              f"train CE={record['train_ce']:.4f} test CE={record['test_ce']:.4f} "
              f"train acc={record['train_acc']:.2f}% "
              f"test acc={record['test_acc']:.2f}%", flush=True)
    print(f"{name}: curve computation completed in {(time.perf_counter()-begin)/60:.1f} minutes.")
    del model, small_params, large_params, train_loader, test_loader, cal_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result, original_endpoint_metrics

## Preflight: ensure F1/C1 measurement files and C2 trained checkpoints exist

In [ ]:
# Recovery preflight: do not run full training, or change experiment assumptions.
import csv, json, os, math, time, gc
from collections import OrderedDict

METRIC_KEYS = ("network", "alpha", "train_ce", "test_ce", "train_acc", "test_acc")
ALPHAS = np.unique(np.concatenate([
    np.linspace(CONFIG["alpha_min"], CONFIG["alpha_max"], CONFIG["alpha_grid_points"]),
    np.asarray([CONFIG["alpha_sb"], CONFIG["alpha_lb"]], dtype=np.float64),
]))


def curve_file(name):
    return OUT_DIR / f"{name}_freshly_measured_curve.csv"


def curve_key(a):
    return f"{float(a):.12f}"


def load_measured_csv(name):
    path = curve_file(name)
    if not path.exists():
        raise FileNotFoundError(f"Missing previously measured curve: {path}")
    records = {}
    with path.open(newline="") as handle:
        reader = csv.DictReader(handle)
        if not reader.fieldnames or not set(METRIC_KEYS).issubset(reader.fieldnames):
            raise ValueError(f"Incorrect curve column names in {path}: {reader.fieldnames}")
        for row in reader:
            if row["network"] != name:
                raise ValueError(f"Found network {row['network']} inside {path}")
            parsed = {"network": name}
            for k in METRIC_KEYS[1:]:
                parsed[k] = float(row[k])
                if not math.isfinite(parsed[k]):
                    raise ValueError(f"Nonfinite {k} in {path}")
            key = curve_key(parsed["alpha"])
            if key in records:
                raise ValueError(f"Duplicate alpha {key} in {path}")
            records[key] = parsed
    valid_keys = {curve_key(alpha) for alpha in ALPHAS}
    if not set(records).issubset(valid_keys):
        raise ValueError(f"Alpha values in {path} do not match the CONFIG grid")
    return records


def save_measured_csv(name, record_map):
    path = curve_file(name)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(METRIC_KEYS))
        writer.writeheader()
        for row in sorted(record_map.values(), key=lambda r: r["alpha"]):
            writer.writerow({k: row[k] for k in METRIC_KEYS})
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


print("Device:", DEVICE, "| GPU available:", torch.cuda.is_available())
if DEVICE.type != "cuda":
    print("WARNING: CPU detected. C2 evaluation can be substantially slower.")
for name in ("F1", "C1"):
    data = load_measured_csv(name)
    if len(data) != len(ALPHAS):
        raise RuntimeError(f"{name}: saved only {len(data)} of {len(ALPHAS)} points. Do not rerun C2 yet.")
    print(f"PRESERVED {name}: {len(data)} measured alpha points in Drive")
for regime in ("SB", "LB"):
    p = ckpt_path("C2", regime, "best")
    if not p.exists():
        raise FileNotFoundError(p)
    print("C2 trained checkpoint:", p)
print("Preflight passed. The next cell runs C2 ONLY and autosaves every new alpha.")

## Evaluate C2 only, with checkpointed interpolation points

In [ ]:
# C2 ONLY. Autosave every alpha to Drive; resume if a prior evaluation was interrupted.
# F1 and C1 are NEVER recomputed here.
name = "C2"
existing = load_measured_csv(name) if curve_file(name).exists() else {}
print(f"C2: {len(existing)}/{len(ALPHAS)} saved. Missing: {len(ALPHAS)-len(existing)}")

if len(existing) < len(ALPHAS):
    model = new_model(name).to(DEVICE)
    sb, lb = read_and_validate_endpoints(name, model)
    train_data, test_data = datasets_for(name)
    cfg = CONFIG["models"][name]
    train_loader = loader_for(train_data, cfg["eval_batch"])
    test_loader = loader_for(test_data, cfg["eval_batch"])
    calibration_loader = loader_for(
        fixed_calibration_subset(train_data, name), cfg["calib_batch"]
    )
    param_names = tuple(dict(model.named_parameters()))
    small_params = {key: sb["model_state"][key].to(DEVICE) for key in param_names}
    large_params = {key: lb["model_state"][key].to(DEVICE) for key in param_names}

    for index, alpha in enumerate(ALPHAS, start=1):
        key = curve_key(alpha)
        if key in existing:
            continue
        start = time.perf_counter()
        interpolate_parameters_(model, small_params, large_params, float(alpha))
        recalibrate_batchnorm(model, calibration_loader)
        train_metrics = measure_dataset(model, train_loader)
        test_metrics = measure_dataset(model, test_loader)
        row = {
            "network": name, "alpha": float(alpha),
            "train_ce": train_metrics["ce"], "test_ce": test_metrics["ce"],
            "train_acc": train_metrics["accuracy"], "test_acc": test_metrics["accuracy"],
        }
        for metric in METRIC_KEYS[1:]:
            if not math.isfinite(row[metric]):
                raise FloatingPointError(f"Nonfinite {metric} at alpha {alpha}")
        existing[key] = row
        save_measured_csv(name, existing)  # durable AFTER EVERY evaluated alpha
        print(f"C2 {index}/{len(ALPHAS)} alpha={alpha:+.3f} "
              f"train CE={row['train_ce']:.4f} test CE={row['test_ce']:.4f} "
              f"train acc={row['train_acc']:.2f}% test acc={row['test_acc']:.2f}% "
              f"time={time.perf_counter()-start:.1f}s (saved)", flush=True)

    del model, sb, lb, small_params, large_params, train_loader, test_loader, calibration_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

assert len(load_measured_csv("C2")) == len(ALPHAS)
print("C2 finished. All 61 C2 points are saved in Google Drive.")

## Generate Figure 3(a,c,d) from the preserved measured data

In [ ]:
# Plot only *measured data* saved from original F1/C1 and the resumed C2 run.
# Never reruns training; requires all three complete 61-point curves.
all_curves = OrderedDict()
for name in CONFIG["networks"]:
    record_map = load_measured_csv(name)
    if len(record_map) != len(ALPHAS):
        raise RuntimeError(f"Incomplete {name}: {len(record_map)}/{len(ALPHAS)} alpha points")
    all_curves[name] = [record_map[curve_key(alpha)] for alpha in ALPHAS]


def plot_measured_figure3(curves, style, config):
    fig, axs = plt.subplots(nrows=style["rows"], ncols=len(curves),
                            figsize=style["figsize"], layout="constrained")
    for ax, (name, records) in zip(np.ravel(axs), curves.items()):
        alpha = [r["alpha"] for r in records]
        acc_ax = ax.twinx()
        ax.plot(alpha, [r["train_ce"] for r in records],
                color=style["loss_color"], linestyle=style["train_ls"],
                linewidth=style["line_width"], label="Train CE")
        ax.plot(alpha, [r["test_ce"] for r in records],
                color=style["loss_color"], linestyle=style["test_ls"],
                linewidth=style["line_width"], label="Test CE")
        acc_ax.plot(alpha, [r["train_acc"] for r in records],
                    color=style["accuracy_color"], linestyle=style["train_ls"],
                    linewidth=style["line_width"], label="Train accuracy")
        acc_ax.plot(alpha, [r["test_acc"] for r in records],
                    color=style["accuracy_color"], linestyle=style["test_ls"],
                    linewidth=style["line_width"], label="Test accuracy")
        ax.set_title(f"{name}: {config['models'][name]['dataset']}")
        ax.set_xlabel(r"$\alpha$ (SB at {sb}, LB at {lb})".format(
            sb=config["alpha_sb"], lb=config["alpha_lb"]))
        ax.set_ylabel("Mean cross-entropy", color=style["loss_color"])
        acc_ax.set_ylabel("Accuracy (%)", color=style["accuracy_color"])
        acc_ax.set_ylim(*style["accuracy_ylim"])
        ax.set_xlim(config["alpha_min"], config["alpha_max"])
        ax.grid(alpha=style["grid_alpha"])
        left_lines, left_labels = ax.get_legend_handles_labels()
        right_lines, right_labels = acc_ax.get_legend_handles_labels()
        ax.legend(left_lines + right_lines, left_labels + right_labels, loc="best")
    return fig

fig = plot_measured_figure3(all_curves, PLOT_STYLE, CONFIG)
plot_path = OUT_DIR / PLOT_STYLE["file_name"]
fig.savefig(plot_path, dpi=PLOT_STYLE["dpi"])
plt.show()
with (OUT_DIR / "experiment_config.json").open("w") as stream:
    json.dump({"training_and_evaluation": CONFIG, "plot_style": PLOT_STYLE,
               "recovery": "F1/C1 evaluated originally; C2 evaluated with per-alpha autosave"},
              stream, indent=2)
print("Final figure saved:", plot_path)
for name in all_curves:
    print(f"{name}: {len(all_curves[name])} measured points from {curve_file(name)}")

## Optional: diagnose calibrated endpoints

The original F1/C1 checkpoint statistics are saved in `F1_original_endpoint_metrics.json` and `C1_original_endpoint_metrics.json` only if your first rescue cell captured them. These files are not needed to construct the figure. To obtain separate raw checkpoint endpoint comparisons for C2, use the original notebook's numerical-endpoint routines (which require full train/test evaluations). The values in the final figure remain actual measurements of the interpolated models and recalibrated BN statistics.